# InvestAgents — bloc D : agents extracteurs de facteurs de risque (Qwen2.5 7B, T4 x2)

Grille figée de 19 facteurs factuels ; citations par numéros de phrases ; « not_found » autorisé.
Vérité terrain (notes IEG) **absente** : elle reste sur le PC. Identifiants et nom des projets
masqués ; pas de recherche web pour les agents (internet sert à installer Ollama et le modèle).

In [ ]:
import subprocess, time, os, glob, json
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
T0 = time.time()

In [ ]:
# Installation d'Ollama (zstd nécessaire à l'archive de l'installeur)
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -3
!ollama --version

In [ ]:
# Un serveur Ollama par GPU
import requests
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.split()
gpus = [g for g in gpus if g == "GPU"]  # une ligne « GPU i: ... » par carte
N = max(1, len(gpus))
URLS, procs = [], []
for i in range(N):
    port = 11434 + i
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), OLLAMA_HOST=f"127.0.0.1:{port}",
               OLLAMA_KEEP_ALIVE="24h", OLLAMA_NUM_PARALLEL="1")
    log = open(f"/kaggle/working/ollama_{i}.log", "w")
    procs.append(subprocess.Popen(["ollama", "serve"], env=env, stdout=log, stderr=log))
    URLS.append(f"http://127.0.0.1:{port}/api/chat")
for i in range(N):
    for _ in range(60):
        try:
            requests.get(f"http://127.0.0.1:{11434 + i}/api/version", timeout=2); break
        except Exception:
            time.sleep(2)
print("serveurs :", URLS)
!OLLAMA_HOST=127.0.0.1:11434 ollama pull qwen2.5:7b 2>&1 | tail -2
for u in URLS:  # chargement du modèle sur chaque GPU
    t = time.time()
    msg = [{"role": "user", "content": "Say OK"}]
    body = {"model": "qwen2.5:7b", "messages": msg, "stream": False, "options": {"num_predict": 5}}
    r = requests.post(u, json=body, timeout=900)
    print(u, r.status_code, r.json()["message"]["content"][:20], f"{time.time()-t:.0f} s")
!nvidia-smi --query-gpu=index,memory.used --format=csv

In [ ]:
# Code de l'extracteur
import tarfile, sys
data_dir = os.path.dirname(glob.glob("/kaggle/input/**/projects.jsonl", recursive=True)[0])
print("données :", data_dir, os.listdir(data_dir))
found = glob.glob("/kaggle/input/**/investagents/extract/runner.py", recursive=True)
if found:
    code_root = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
else:
    tarfile.open(os.path.join(data_dir, "investagents_code.tar.gz")).extractall("/tmp/code")
    code_root = "/tmp/code"
sys.path.insert(0, code_root)
import investagents.extract.runner
print("code OK :", code_root, f"{(time.time()-T0)/60:.1f} min")

In [ ]:
# Extraction (pilot) ; sauvegarde après chaque projet ; arrêt propre avant 12 h
deadline = 11.0 - (time.time() - T0) / 3600
prev = " ".join(glob.glob("/kaggle/input/**/results.jsonl", recursive=True))
resume = f"--resume-from {prev}" if prev else ""
urls = " ".join(URLS)
args = f"--data {data_dir} --out /kaggle/working --urls {urls} --deadline-hours {deadline}"
args += f" {resume} --model qwen2.5:7b --limit 30"
!cd /tmp && PYTHONPATH={code_root} python -u -m investagents.extract.runner {args}
print(open("/kaggle/working/progress.json").read())
for p in procs:
    p.terminate()